# Lecture 03: Experiment Tracking with Weights & Biases

This notebook tracks the training of Model A with Weights & Biases (W&B). Every training becomes a **run** that stores its settings, its metrics, its plots and its model file. At the end you register the best model, so that your backend can load it by name.

W&B is a hosted service. You need an account, which is free for academic use with your university email address. Since 30 September 2026 W&B is part of CoreWeave Forge, and you sign in there with your W&B login. The `wandb` library stays the same.

Logistic regression serves as the example.

In [ ]:
!uv pip install pandas scikit-learn wandb joblib

## 1. Log in once

Log in once per machine. W&B asks for your API key and stores it in your home folder, outside the repository. Never write the key into a notebook or a script, and never commit it.

You can log in from a terminal with `uv run wandb login`, or run the next cell, which asks for the key.

Without an internet connection, set the environment variable `WANDB_MODE=offline` before you start. W&B then writes the runs to a local folder, and `wandb sync` uploads them later.

In [ ]:
import wandb

wandb.login()

## 2. The data

The same cleaned data and the same split as in notebook 02. The hash of the CSV file identifies the exact version of the data. It goes into the config of every run.

In [ ]:
import hashlib

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT = "credit-model-a"
DATA = "../../data/credit_risk_dataset.csv"
SEED = 42

raw = pd.read_csv(DATA).drop_duplicates()
late_cols = ["late_30_59", "late_60_89", "late_90"]
df = raw[~raw[late_cols].isin([96, 98]).any(axis=1)]
X, y = df.drop(columns="dlq_2yrs"), df["dlq_2yrs"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
DATA_HASH = hashlib.sha256(open(DATA, "rb").read()).hexdigest()[:12]
print(len(X_train), len(X_val), len(X_test), DATA_HASH)

## 3. Log the dataset as an artifact

An **artifact** is a versioned file in W&B. Log the dataset once. Every training run then declares which version it used, and W&B links data, runs and models.

In [ ]:
with wandb.init(project=PROJECT, job_type="upload-data", name="credit-data") as run:
    data_artifact = wandb.Artifact("credit-data", type="dataset", metadata={"rows": len(raw), "sha256": DATA_HASH})
    data_artifact.add_file(DATA)
    run.log_artifact(data_artifact)

## 4. A tracked training run

The function `train` trains one model and records what a teammate needs to repeat it: the settings and seeds in `config`, the data version, the metrics, three plots and the model file. All metrics come from the validation set. The test set stays untouched.

In [ ]:
def evaluate(model, threshold):
    p_val = model.predict_proba(X_val)[:, 1]
    pred = (p_val >= threshold).astype(int)
    metrics = {"val/accuracy": accuracy_score(y_val, pred),
               "val/precision": precision_score(y_val, pred, zero_division=0),
               "val/recall": recall_score(y_val, pred),
               "val/f1": f1_score(y_val, pred, zero_division=0),
               "val/mcc": matthews_corrcoef(y_val, pred),
               "val/auc": roc_auc_score(y_val, p_val),
               "val/ap": average_precision_score(y_val, p_val)}
    return p_val, pred, metrics


def build(config):
    if config["model"] == "baseline":
        return DummyClassifier(strategy="most_frequent")
    weight = None if config["class_weight"] == "none" else config["class_weight"]
    return make_pipeline(StandardScaler(), LogisticRegression(C=config["C"], class_weight=weight, max_iter=1000))


def train(config=None, name=None, tags=(), group=None):
    defaults = {"model": "logistic_regression", "C": 1.0, "class_weight": "none", "threshold": 0.5,
                "split_seed": SEED, "features": list(X.columns), "data_sha256": DATA_HASH,
                "rows_train": len(X_train), "rows_val": len(X_val)}
    with wandb.init(project=PROJECT, config={**defaults, **(config or {})}, name=name, tags=list(tags),
                    group=group, job_type="train") as run:
        if not run.offline:   # offline runs cannot fetch artifacts
            run.use_artifact("credit-data:latest")
        cfg = dict(run.config)
        model = build(cfg).fit(X_train, y_train)
        p_val, pred, metrics = evaluate(model, cfg["threshold"])
        run.log(metrics)
        probas = np.c_[1 - p_val, p_val].tolist()
        # 1 - label makes "default" the first class, so TP sits top left as on the slides
        run.log({"val/confusion_matrix": wandb.plot.confusion_matrix(y_true=(1 - y_val).tolist(), preds=(1 - pred).tolist(),
                                                                     class_names=["default", "no default"]),
                 "val/roc": wandb.plot.roc_curve(y_val.tolist(), probas, labels=["no default", "default"],
                                                 classes_to_plot=[1]),
                 "val/pr": wandb.plot.pr_curve(y_val.tolist(), probas, labels=["no default", "default"],
                                               classes_to_plot=[1])})
        if cfg["model"] != "baseline":
            joblib.dump(model, "model.joblib")
            model_artifact = wandb.Artifact("model-a", type="model", metadata={"val_auc": metrics["val/auc"]})
            model_artifact.add_file("model.joblib")
            run.log_artifact(model_artifact)
        print(run.name, {k: round(v, 3) for k, v in metrics.items()})

Track the simplest baseline and two logistic regressions. Each call creates one run.

In [ ]:
train({"model": "baseline"}, name="baseline-most-frequent", tags=["baseline"], group="baseline")
train({"C": 1.0}, name="logreg-C1", tags=["logreg"], group="logistic-regression")
train({"C": 1.0, "class_weight": "balanced"}, name="logreg-C1-balanced", tags=["logreg"], group="logistic-regression")

## 5. Compare the runs in the dashboard

Open your project with the link that W&B printed above. In the runs table, show the column of your metric, sort by it and group the runs by `model`. Each run links to its config, its plots and its model file. If you run the code from a Git repository, W&B also records the commit.

## 6. A sweep

A sweep starts many runs with different settings and keeps the best by the metric you name. `val/auc` below is only an example. Replace it with the metric you chose and defended in your decision log.

In [ ]:
METRIC = "val/auc"

sweep = {"method": "bayes",
         "metric": {"name": METRIC, "goal": "maximize"},
         "parameters": {"C": {"distribution": "log_uniform_values", "min": 0.001, "max": 100.0},
                        "class_weight": {"values": ["none", "balanced"]}}}

sweep_id = wandb.sweep(sweep, project=PROJECT)
wandb.agent(sweep_id, function=lambda: train(tags=["sweep"], group="sweep-logreg"), count=15)

Look at the parallel coordinates chart of the sweep. For logistic regression on this data, `C` hardly changes the result. The settings of your random forest matter much more.

## 7. Register the best model

Choose the best run in the dashboard and look up the version of its model artifact, for example `model-a:v3`. The cell links this version to the **Models** registry under the name `model-a`, with the alias `candidate`.

Moving the alias `production` to a version is a decision. Make it in the W&B interface, and record it in your decision log.

In [ ]:
BEST_MODEL = "model-a:v0"

with wandb.init(project=PROJECT, job_type="register") as run:
    best = run.use_artifact(BEST_MODEL)
    run.link_artifact(best, target_path="wandb-registry-model/model-a", aliases=["candidate"])

## 8. Load the model by alias

This is how your backend loads the model. It asks for an alias, not for a file. In the backend, ask for `production`. The backend needs an API key at run time. Pass it through the environment variable `WANDB_API_KEY`, never through the code.

In [ ]:
api = wandb.Api()
registered = api.artifact("wandb-registry-model/model-a:candidate")
loaded = joblib.load(f"{registered.download()}/model.joblib")
print(loaded.predict_proba(X_val.head(3))[:, 1])